# Module C, sequence 3: analysis (exercises)

Replace each `...` by what its comment asks, then run the cell. The steps follow the TP 3 of the handout of the module C. The corrected notebook is in the branch `correction-module-c`.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from pymodaq_utils.math_utils import gauss1D
from pymodaq_data.h5modules.data_saving import DataLoader

DATA = Path('../data')  # a pathlib.Path: the operator / joins path parts (DATA / 'file.h5')
SCAN = '/RawData/Scan000/Detector000/Data0D/CH00/Data00'  # one scan
SCAN8 = '/RawData/Scan001/Detector000/Data0D/CH00/Data00'  # the same scan repeated 8 times (scan_average = 8)


def model(x, amp, x0, fwhm, offset):
    return amp * gauss1D(x, x0, fwhm) + offset  # FWHM is the width parameter of gauss1D

## 1. Load the scan file

In [ ]:
with DataLoader(DATA / 'scan_photodiode.h5') as loader:
    scan = ...  # the data of SCAN, with its navigation axes (hint: loader.load_data(..., load_all=True))
    scan8 = ...  # the data of SCAN8, the same way
print(scan, scan.axes)  # compare the shapes and the axes: the Average axis is only in the second one
print(scan8, scan8.axes)

## 2. Look at it and find the lines

In [ ]:
mean8 = ...  # the mean over the first navigation axis, the repetitions (hint: scan8.mean)
wavelength = ...  # the values of the wavelength axis of mean8 (hint: get_axis_from_index, get_data)
fig, ax = plt.subplots()
ax.plot(wavelength, scan.data[0], '.', label='one scan')
ax.plot(wavelength, mean8.data[0], '-', label='mean of 8 scans')
ax.set_xlabel('wavelength (nm)')
ax.legend()
peaks, _ = find_peaks(mean8.data[0], height=1.0)
print(wavelength[peaks])  # about 471, 529 and 590 nm

## 3. Crop around the main line and fit with scipy (uncertainties from the covariance)

In [ ]:
window = ...  # mean8 between 505 and 555 nm (hint: vnav: by values, on the navigation axis)
...  # declare the navigation axis as a signal axis: `window` becomes a 1D data (hint: nav_indexes)
x = window.get_axis_from_index(0)[0].get_data()
y = window.data[0]
p0 = [y.max() - y.min(), x[np.argmax(y)], 5., y.min()]
popt, pcov = ...  # fit `model` to (x, y) starting from p0 (hint: curve_fit)
perr = ...  # the uncertainties of the parameters: square root of the diagonal of pcov
for name, value, error in zip(['amplitude', 'center (nm)', 'FWHM (nm)', 'offset'], popt, perr):
    print(f'{name}: {value:.3f} +/- {error:.3f}')

## 4. The same fit with DataWithAxes.fit

In [ ]:
fit = ...  # fit `window` with `model`, starting from p0, with the method of the data object (hint: window.fit)
print(fit.fit_coeffs[0])
assert np.allclose(fit.fit_coeffs[0], popt, rtol=1e-3)  # what is missing compared to curve_fit?

## 5. The axis in eV

In [ ]:
axis_ev = ...  # the axis of `window` converted to eV, without modifying it (hint: units_as, context 'spectroscopy')
window_ev = window.deepcopy()
window_ev.axes[0] = axis_ev
x_ev = axis_ev.get_data()
y_ev = window_ev.data[0]
p0_ev = [10., x_ev[np.argmax(y_ev)], 0.02, y_ev.min()]
popt_ev, pcov_ev = curve_fit(model, x_ev, y_ev, p0=p0_ev)
print(axis_ev.units, popt_ev[1], popt_ev[2], np.sqrt(np.diag(pcov_ev))[1:3])  # centre and FWHM in eV

## 6. One spectrum with Naverage = 200: the errors come with the data (PyMoDAQ 5.3.9 or later)

In [ ]:
with DataLoader(DATA / 'spectrum_naverage200.h5') as loader:
    spectrum = ...  # load everything under '/RawData/Detector000' and keep the data named 'Spectrum' (hint: load_all, get_data_from_name)
print(spectrum.errors[0][:3])

Plot the spectrum with its errors in the PyMoDAQ viewer (needs a display): where is the band the widest?

In [ ]:
%gui qt
import pymodaq_gui  # registers the Qt plotter
...  # plot `spectrum` in the PyMoDAQ viewer (hint: plot with the backend 'qt')

## 7. Going further (open, for those who are done)

No hole to fill here and no single answer: pick one or several items, write your own cells, and keep what you find
in your handout. Everything you need is in the files of the `data` folder and in the objects above.

1. **The whole scan.** Fit `mean8` without cropping (three gaussians and one background, or one gaussian and the
   whole scan). Compare the background with the one of the cropped fit of the part 3: what do the satellite lines do?
   You will meet a point that the cropped window did not contain: look at `np.isnan(scan8.data[0])` and at what
   `curve_fit` says.
2. **Uncertainties from the repetitions.** `scan8.mean_with_errors(0)` puts the standard error of the mean of the 8
   repetitions in `errors`. Crop it again (the errors follow the slicing), plot it with `errorbar`, fit it with
   `curve_fit(..., sigma=window.errors[0], absolute_sigma=True)` and compute the reduced chi2. If it is larger than 1,
   are the error bars too small or is the model incomplete?
3. **Two ways to get a standard error.** Load `continuous_spectra.h5`
   (`'/RawData/Detector000/Data1D/CH00/EnlData00'`) and compare `series.mean_with_errors(0).errors[0]` with the
   `errors` of the spectrum of the part 6, at the top of the line and far from it. They are close at both places but
   much smaller far from the line: why? What does the standard deviation at the top, compared with the one far from
   the line, say about the origin of the noise?
4. **Model the drift.** In the continuous recording, fit the intensity at the peak versus the acquisition number with
   a sinusoid. What are its period (in acquisitions) and its relative depth? Divide the series by the fitted drift
   and compute again the standard deviation at the peak: what is left?
5. **The fit as a data object.** Build a `DataCalculated` holding the data of `window` and the fitted curve as two
   channels (same axis, labels 'data' and 'fit'), plot it with the PyMoDAQ viewer (`import pymodaq_gui` first, then
   `%gui qt` and `.plot('qt')`), then save it with its errors in an HDF5 file and open it in H5Browser.
6. **Navigation and signal on the series.** `series` has a navigation axis (time) and a signal axis (wavelength).
   Keep 525 to 532 nm on the signal axis with `vsig`: why does PyMoDAQ refuse (look at `series.distribution`)? Do it
   with `isig` instead (the axis has a `find_index` method), sum over the signal axis to get one number per
   acquisition, and compare with the intensity at the peak. Then keep the first 50 acquisitions only.
7. **Your own measurement.** Record a scan of the photodiode yourself in the DAQ_Scan (module A), with the satellite
   lines and the drift switched on in the settings of the plugin, and run this notebook on your file.